# Module 2 -- Step 4: Test Model Access

This notebook demonstrates **four ways** to interact with the LLM Gateway:

1. Direct HTTP calls with `requests`
2. OpenAI SDK compatibility
3. Strands Agent integration (via LiteLLM)
4. Workshop client library

Along the way we also verify health, list available models, exercise multi-model routing, and observe response caching.

In [ ]:
import json, time, requests

with open(".state.json") as f:
    state = json.load(f)

LLM_GATEWAY_URL = state["LLM_GATEWAY_URL"]
API_KEY = state["API_KEY"]
ADMIN_KEY = state["ADMIN_KEY"]
AWS_REGION = state["AWS_REGION"]

print(f"Gateway: {LLM_GATEWAY_URL}")

## 1. Health Check

A quick liveness probe to confirm the gateway is up and reachable.

In [ ]:
resp = requests.get(f"{LLM_GATEWAY_URL}/health/liveliness", timeout=5)
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## 2. List Available Models

The gateway exposes an OpenAI-compatible `/models` endpoint that returns every model currently configured for routing.

In [ ]:
headers = {"Authorization": f"Bearer {API_KEY}"}
resp = requests.get(f"{LLM_GATEWAY_URL}/models", headers=headers, timeout=10)
models = resp.json().get("data", [])
print(f"Available models: {len(models)}\n")
for m in models:
    print(f"  {m['id']}")

## 3. Chat Completion via requests

The most direct way to call the gateway -- a plain HTTP POST to the `/chat/completions` endpoint using the `requests` library.

In [ ]:
headers = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"}
resp = requests.post(f"{LLM_GATEWAY_URL}/chat/completions", json={
    "model": "claude-sonnet",
    "messages": [{"role": "user", "content": "What are three benefits of using an LLM Gateway?"}],
    "max_tokens": 300,
}, headers=headers, timeout=30)
resp.raise_for_status()
result = resp.json()
print(result["choices"][0]["message"]["content"])
print(f"\nTokens: {result['usage']}")

## 4. OpenAI SDK Compatibility

Because the gateway implements the OpenAI Chat Completions API contract, you can point the official `openai` Python SDK at it by overriding `base_url`. Any code already written for OpenAI works with zero changes -- just swap the URL and key.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=LLM_GATEWAY_URL)
response = client.chat.completions.create(
    model="claude-sonnet",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "Explain what an LLM Gateway does in 2 sentences."},
    ],
    max_tokens=150,
)
print(response.choices[0].message.content)
print(f"\nModel: {response.model}")
print(f"Tokens: prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens}")

## 5. Multi-Model Routing

One of the key advantages of an LLM Gateway is **model routing** -- the same API call can target different backend models simply by changing the `model` parameter. Below we send an identical prompt to several models and compare their responses.

In [ ]:
prompt = "In one sentence, what makes you unique as an AI model?"
models_to_test = ["claude-sonnet", "claude-haiku", "nova-lite"]

success_count = 0
failures = []
for model_id in models_to_test:
    try:
        resp = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=100,
        )
        print(f"[{model_id}]")
        print(f"  {resp.choices[0].message.content}\n")
        success_count += 1
    except Exception as e:
        print(f"[{model_id}] Failed: {e}\n")
        failures.append((model_id, str(e)))

if success_count == 0:
    raise RuntimeError(
        f"No models responded successfully — enable model access in the AWS Bedrock "
        f"console (Bedrock > Model access). Failures: {failures}"
    )

## 6. Caching Demonstration

The gateway supports response caching for deterministic requests (`temperature=0`). Sending the same request twice should show a measurable latency improvement on the second call.

In [ ]:
prompt = "What is 2 + 2? Reply with just the number."

start = time.time()
r1 = client.chat.completions.create(model="claude-sonnet", messages=[{"role": "user", "content": prompt}], max_tokens=10, temperature=0)
t1 = time.time() - start
print(f"Request 1: '{r1.choices[0].message.content.strip()}' ({t1:.2f}s)")

start = time.time()
r2 = client.chat.completions.create(model="claude-sonnet", messages=[{"role": "user", "content": prompt}], max_tokens=10, temperature=0)
t2 = time.time() - start
print(f"Request 2: '{r2.choices[0].message.content.strip()}' ({t2:.2f}s)")

# Prefer the header-based signal over a timing heuristic
cache_hit = False
try:
    r2_raw = r2.model_dump() if hasattr(r2, "model_dump") else {}
    cache_hit = bool((r2_raw.get("_hidden_params") or {}).get("cache_hit"))
except Exception:
    cache_hit = False

if cache_hit:
    print(f"\nCache hit confirmed via hidden_params. Second call: {t2:.2f}s vs first: {t1:.2f}s")
elif t2 < t1 * 0.5:
    print(f"\nLikely cache hit (second request {t1/t2:.1f}x faster).")
else:
    print("\nNo cache-hit signal observed. LiteLLM caching may not be enabled on this "
          "deployment — this is expected. Both calls take similar time.")

## 7. Strands Agent Integration

Strands agents can use the LLM Gateway as their model backend via the LiteLLM adapter. This lets you build agentic workflows while retaining all gateway benefits -- routing, caching, observability, and access controls.

In [ ]:
from strands import Agent
from strands.models.litellm import LiteLLMModel

model = LiteLLMModel(
    # Route through the gateway: connection details go in client_args
    # (api_key + base_url, which litellm.LiteLLM accepts); the "openai/"
    # model_id prefix tells LiteLLM to use the OpenAI-compatible proxy path.
    client_args={"api_key": API_KEY, "base_url": LLM_GATEWAY_URL},
    model_id="openai/claude-sonnet",
)

agent = Agent(model=model)
result = agent("What is 15 * 23? Think step by step and give the answer.")
print(f"Agent response: {result}")

## 8. Workshop Client Library

The workshop includes a thin client library (`llm_gateway_client`) that wraps the gateway API with typed Pydantic models and convenience methods.

In [ ]:
import sys, os
# Resolve module-2-llm-gateway root regardless of notebook launch dir
for candidate in (os.path.abspath(os.path.join(os.getcwd(), "..")), os.getcwd()):
    if os.path.isdir(os.path.join(candidate, "llm_gateway_client")) or os.path.isfile(os.path.join(candidate, "llm_gateway_client.py")):
        sys.path.insert(0, candidate)
        break
from llm_gateway_client import LLMGatewayClient

gw = LLMGatewayClient(proxy_url=LLM_GATEWAY_URL, api_key=API_KEY)

# Quick chat
response = gw.chat("What is the capital of France?")
print(f"Chat: {response}")

# List models
models = gw.list_models()
print(f"\nModels: {len(models.data)}")
for m in models.data[:5]:
    print(f"  {m.id}")

---

## Next Step

Proceed to **Step 5** to set up enterprise guardrails using Amazon Bedrock
content filters and test them through the LLM Gateway.

Open **`step-5-guardrails.ipynb`** to continue.